## Diffusion Model & Flow Matching Tutorial
### Problem Set 2: DDPM, Flow Matching, and Mean Flow


In [ ]:
import torch
import torch.nn as nn
from IPython.display import clear_output
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import math
import os
os.makedirs('result', exist_ok=True)


### Setup: MLP Backbone


In [ ]:
class MLP_Backbone(nn.Module):
    def __init__(self, n_steps, input_dim=2):
        super().__init__()
        self.linear_model1 = nn.Sequential(
            nn.Linear(input_dim, 256), nn.Dropout(0.2), nn.GELU()
        )
        self.embedding_layer = nn.Embedding(n_steps, 256)
        self.linear_model2 = nn.Sequential(
            nn.Linear(256, 512), nn.Dropout(0.2), nn.GELU(),
            nn.Linear(512, 512), nn.Dropout(0.2), nn.GELU(),
            nn.Linear(512, input_dim),
        )
    def forward(self, x, idx):
        x = self.linear_model2(self.linear_model1(x) + self.embedding_layer(idx))
        return x


### Setup: DDPM Class


In [ ]:
class DDPM(nn.Module):
    def __init__(self, device, beta_1, beta_T, T, backbone, shape):
        super().__init__()
        self.device = device
        self.T = T
        self.betas = torch.linspace(start=beta_1, end=beta_T, steps=T).to(self.device)
        self.alphas = (1 - self.betas).to(self.device)
        self.alpha_bars = torch.cumprod(1 - torch.linspace(start=beta_1, end=beta_T, steps=T), dim=0).to(self.device)
        self.alpha_prev_bars = torch.cat([torch.Tensor([1]).to(device=device), self.alpha_bars[:-1]]).to(self.device)
        self.shape = shape
        self.backbone = backbone.to(self.device)

    def forward_process(self, x0, t):
        epsilon = torch.randn_like(x0)
        if len(x0.shape) == 2:
            alpha_bars = self.alpha_bars[t][:, None]
        elif len(x0.shape) == 4:
            alpha_bars = self.alpha_bars[t][:, None, None, None]
        # ================ Your Implementation Start ==========================
        raise NotImplementedError
        # ================ Your Implementation End ==========================
        return x_t, epsilon

    def loss_fn(self, x0):
        t = torch.randint(0, len(self.alpha_bars), (x0.size(0),)).to(device=self.device)
        x_t, epsilon = self.forward_process(x0, t)
        # ================ Your Implementation Start ==========================
        raise NotImplementedError
        # ================ Your Implementation End ==========================
        return loss

    @torch.no_grad()
    def sampling(self, sampling_number, only_final=False):
        sample = torch.randn([sampling_number, *self.shape]).to(device=self.device)
        sampling_list = []
        final = None
        for idx, sample in enumerate(self.reverse_process(sample)):
            final = sample
            if not only_final:
                sampling_list.append(final)
        return final if only_final else torch.stack(sampling_list)

    def reverse_process(self, xt):
        for t in reversed(range(len(self.alpha_bars))):
            noise = torch.zeros_like(xt) if t == 0 else torch.randn_like(xt)
            time = torch.Tensor([t for _ in range(xt.size(0))]).to(self.device).long()
            # ================ Your Implementation Start ==========================
            raise NotImplementedError
            # ================ Your Implementation End ============================
            yield xt


In [ ]:
class AverageMeter(object):
    def __init__(self, name, fmt=':f'):
        self.name = name; self.fmt = fmt; self.reset()
    def reset(self):
        self.val = 0; self.avg = 0; self.sum = 0; self.count = 0
    def update(self, val, n=1):
        self.val = val; self.sum += val * n; self.count += n; self.avg = self.sum / self.count
    def __str__(self):
        fmtstr = '{name} {val' + self.fmt + '} ({avg' + self.fmt + '})'
        return fmtstr.format(**self.__dict__)

class ProgressMeter(object):
    def __init__(self, num_batches, meters, prefix=''):
        self.batch_fmtstr = self._get_batch_fmtstr(num_batches)
        self.meters = meters; self.prefix = prefix
    def display(self, batch):
        entries = [self.prefix + self.batch_fmtstr.format(batch)]
        entries += [str(meter) for meter in self.meters]
        print('\r' + '\t'.join(entries), end='')
    def _get_batch_fmtstr(self, num_batches):
        num_digits = len(str(num_batches // 1))
        fmt = '{' + ':' + str(num_digits) + 'd}'
        return '[' + fmt + '/' + fmt.format(num_batches) + ']'

def scatter(sample, only_final, scatter_range=[-10, 10]):
    clear_output()
    if only_final:
        s = sample.detach().cpu().numpy()
        plt.figure(figsize=(7, 7)); plt.xlim(scatter_range); plt.ylim(scatter_range)
        plt.scatter(s[:, 0], s[:, 1], s=5); plt.show()
    else:
        step_size = sample.size(0)
        fig, axs = plt.subplots(1, step_size, figsize=(step_size * 4, 4), constrained_layout=True)
        for i in range(step_size):
            s = sample[i].detach().cpu().numpy()
            axs[i].scatter(s[:, 0], s[:, 1], s=5)
            axs[i].set_xlim(scatter_range); axs[i].set_ylim(scatter_range)
        plt.show()


## Problem 1: DDPM


In [ ]:
class Ring_Dataset(torch.utils.data.Dataset):
    def __init__(self, total_len=1000000, radius=5.0, std=0.1):
        self.total_len = total_len; self.radius = radius; self.std = std
    def __len__(self):
        return self.total_len
    def __getitem__(self, idx):
        theta = np.random.uniform(0, 2 * np.pi)
        x = self.radius * np.cos(theta); y = self.radius * np.sin(theta)
        noise = np.random.normal(0, self.std, 2)
        return torch.tensor([x, y], dtype=torch.float32) + torch.tensor(noise, dtype=torch.float32)


In [ ]:
batch_size = 8192
vis_dataloader = torch.utils.data.DataLoader(Ring_Dataset(total_len=batch_size), batch_size=batch_size, drop_last=True)
scatter(next(iter(vis_dataloader)), True)


In [ ]:
# ============ Don't change the parameters ============
beta_1 = 1e-4; beta_T = 0.02; T = 100; shape = (2,)
# =====================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mlp_backbone = MLP_Backbone(n_steps=T, input_dim=shape[0])
ddpm = DDPM(device, beta_1, beta_T, T, backbone=mlp_backbone, shape=shape).to(device)
optim = torch.optim.Adam(ddpm.parameters(), lr=0.005)


In [ ]:
"""DDPM Training"""
total_iteration = 1000; batch_size = 8192
losses = AverageMeter('Loss', ':.4f')
progress = ProgressMeter(total_iteration, [losses], prefix='Iteration')
train_dataloader = torch.utils.data.DataLoader(Ring_Dataset(total_len=batch_size * total_iteration), batch_size=batch_size, drop_last=True)
current_iteration = 0
for data in train_dataloader:
    data = data.to(device=device)
    loss = ddpm.loss_fn(data)
    optim.zero_grad(); loss.backward(); optim.step()
    losses.update(loss.item()); progress.display(current_iteration)
    current_iteration += 1
    if current_iteration >= total_iteration: break
print('\nFinished Training')


In [ ]:
sampling_number = 80000
sample = ddpm.sampling(sampling_number, only_final=True)
scatter(sample, True)


In [ ]:
sampling_number = 80000; scatter_range = [-10, 10]
sample = ddpm.sampling(sampling_number, only_final=False)
scatter(sample[9::10], False)

def update_plot(i, data, scat):
    scat.set_offsets(data[i].detach().cpu().numpy()); return scat

numframes = len(sample)
fig = plt.figure(figsize=(6, 6)); plt.xlim(scatter_range); plt.ylim(scatter_range)
scat = plt.scatter(sample[0].detach().cpu().numpy()[:, 0], sample[0].detach().cpu().numpy()[:, 1], s=1)
plt.show(); clear_output()
ani = animation.FuncAnimation(fig, update_plot, frames=range(numframes), fargs=(sample, scat), interval=50)
writergif = animation.PillowWriter(fps=40)
ani.save('result/ddpm_ring.gif', writer=writergif)
HTML(ani.to_jshtml())


---
## Problem 2: Conditional Flow Matching (CFM)
Train a neural velocity field to transport Gaussian noise to the 8-Gaussians distribution.


In [ ]:
class EightGaussians_Dataset(torch.utils.data.Dataset):
    def __init__(self, total_len=1000000, radius=5.0, std=0.2):
        self.total_len = total_len; self.radius = radius; self.std = std
    def __len__(self):
        return self.total_len
    def __getitem__(self, idx):
        k = np.random.randint(0, 8); theta = k * np.pi / 4
        center = np.array([self.radius * np.cos(theta), self.radius * np.sin(theta)])
        return torch.tensor(center + np.random.randn(2) * self.std, dtype=torch.float32)

batch_size = 8192
vis_data = next(iter(torch.utils.data.DataLoader(EightGaussians_Dataset(total_len=batch_size), batch_size=batch_size)))
plt.figure(figsize=(7, 7))
plt.scatter(vis_data[:, 0].numpy(), vis_data[:, 1].numpy(), s=2, alpha=0.5)
plt.xlim([-10, 10]); plt.ylim([-10, 10]); plt.title('8-Gaussians Distribution'); plt.show()


In [ ]:
class VelocityMLP(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim + 1, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, input_dim),
        )
    def forward(self, x, t):
        if t.dim() == 1: t = t.unsqueeze(-1)
        return self.net(torch.cat([x, t], dim=-1))


In [ ]:
def cfm_loss(model, x1, device):
    """Compute the Conditional Flow Matching loss.
    Steps:
        1. Sample x0 ~ N(0, I) with same shape as x1
        2. Sample t ~ Uniform[0, 1] with shape (batch,)
        3. Compute x_t = (1 - t) * x0 + t * x1
        4. Target velocity: u_t = x1 - x0
        5. Predict velocity: v = model(x_t, t)
        6. Return MSE loss: mean(||v - u_t||^2)
    """
    # ================ Your Implementation Start ==========================
    raise NotImplementedError
    # ================ Your Implementation End ==========================


In [ ]:
@torch.no_grad()
def euler_ode_solve(model, x0, n_steps=100, device=None):
    """Generate samples by Euler integration from t=0 to t=1.
    Returns: (x, trajectory) where trajectory is a list of states.
    """
    if device is None: device = x0.device
    dt = 1.0 / n_steps; x = x0.clone(); trajectory = [x.clone()]
    # ================ Your Implementation Start ==========================
    # For each step i=0..n_steps-1:
    #   t = i * dt (as tensor of shape (batch,))
    #   v = model(x, t)
    #   x = x + v * dt
    #   trajectory.append(x.clone())
    raise NotImplementedError
    # ================ Your Implementation End ==========================
    return x, trajectory


In [ ]:
velocity_model = VelocityMLP(input_dim=2, hidden_dim=256).to(device)
optim_cfm = torch.optim.Adam(velocity_model.parameters(), lr=1e-3)
total_iteration = 3000; batch_size = 4096
losses = AverageMeter('Loss', ':.4f'); progress = ProgressMeter(total_iteration, [losses], prefix='CFM Training')
train_dataloader = torch.utils.data.DataLoader(EightGaussians_Dataset(total_len=batch_size * total_iteration), batch_size=batch_size, drop_last=True)
current_iteration = 0
for data in train_dataloader:
    data = data.to(device)
    loss = cfm_loss(velocity_model, data, device)
    optim_cfm.zero_grad(); loss.backward(); optim_cfm.step()
    losses.update(loss.item()); progress.display(current_iteration)
    current_iteration += 1
    if current_iteration >= total_iteration: break
print('\nFinished CFM Training')


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(30, 6)); scatter_range = [-10, 10]
for idx, n_steps in enumerate([1, 5, 10, 50, 100]):
    x0 = torch.randn(10000, 2).to(device)
    samples, _ = euler_ode_solve(velocity_model, x0, n_steps=n_steps)
    s = samples.detach().cpu().numpy()
    axes[idx].scatter(s[:, 0], s[:, 1], s=1, alpha=0.5)
    axes[idx].set_xlim(scatter_range); axes[idx].set_ylim(scatter_range)
    axes[idx].set_title(f'Euler steps = {n_steps}')
plt.tight_layout(); plt.savefig('result/cfm_steps_comparison.png', dpi=150); plt.show()


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 6))
grid_range = np.linspace(-8, 8, 20); xx, yy = np.meshgrid(grid_range, grid_range)
grid_points = torch.tensor(np.stack([xx.flatten(), yy.flatten()], axis=1), dtype=torch.float32).to(device)
for idx, t_val in enumerate([0.0, 0.25, 0.5, 0.75]):
    t = torch.ones(grid_points.size(0), device=device) * t_val
    with torch.no_grad(): v = velocity_model(grid_points, t).cpu().numpy()
    axes[idx].quiver(xx.flatten(), yy.flatten(), v[:, 0], v[:, 1], alpha=0.7)
    axes[idx].set_xlim([-10, 10]); axes[idx].set_ylim([-10, 10]); axes[idx].set_title(f't = {t_val}')
plt.tight_layout(); plt.savefig('result/cfm_velocity_field.png', dpi=150); plt.show()


---
## Problem 3: Mean Flow

The **mean velocity** from time 0 to time $t$ along an ODE trajectory is:

$$\\bar{u}(z_t, t) = \\frac{z_t - z_0}{t}, \\quad t > 0$$

The **MeanFlow Identity** states:

$$\\bar{u}(z_t, t) = v(z_t, t) - t \\cdot \\frac{d\\bar{u}}{dt}$$

### Verification and One-Step Generation
We reuse `euler_ode_solve` from Problem 2 to generate trajectories.


In [ ]:
@torch.no_grad()
def generate_trajectories(model, n_samples, n_steps=100, device=None):
    """Generate ODE trajectories (reuses euler_ode_solve)"""
    if device is None: device = next(model.parameters()).device
    z0 = torch.randn(n_samples, 2).to(device)
    _, trajectory = euler_ode_solve(model, z0, n_steps=n_steps, device=device)
    times = [i / n_steps for i in range(n_steps + 1)]
    return z0, trajectory, times


In [ ]:
"""Verify the MeanFlow Identity numerically"""
z0, trajectory, times = generate_trajectories(velocity_model, n_samples=200, n_steps=100, device=device)
traj = torch.stack(trajectory)
sample_idx = 0; z0_single = z0[sample_idx]; traj_single = traj[:, sample_idx, :]
mean_vel = []; inst_vel = []; valid_times = []
for i in range(1, len(times)):
    t_val = times[i]; z_t = traj_single[i].unsqueeze(0)
    t_tensor = torch.tensor([t_val], device=device)
    u_bar = (z_t.squeeze() - z0_single) / t_val
    mean_vel.append(u_bar.cpu().numpy())
    with torch.no_grad(): v = velocity_model(z_t, t_tensor).squeeze()
    inst_vel.append(v.cpu().numpy()); valid_times.append(t_val)
mean_vel = np.array(mean_vel); inst_vel = np.array(inst_vel); valid_times = np.array(valid_times)
d_mean_vel = np.gradient(mean_vel, valid_times, axis=0)
identity_rhs = inst_vel - valid_times[:, None] * d_mean_vel
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for dim in range(2):
    axes[dim].plot(valid_times, mean_vel[:, dim], label=r'$\bar{u}$ (mean velocity)', linewidth=2)
    axes[dim].plot(valid_times, identity_rhs[:, dim], '--', label=r'$v - t \cdot d\bar{u}/dt$', linewidth=2)
    axes[dim].plot(valid_times, inst_vel[:, dim], ':', label=r'$v$ (instantaneous)', alpha=0.7)
    axes[dim].set_xlabel('t'); axes[dim].set_ylabel(f'Velocity (dim {dim})')
    axes[dim].legend(); axes[dim].set_title(f'MeanFlow Identity Verification (dim {dim})')
plt.tight_layout(); plt.savefig('result/meanflow_identity_verification.png', dpi=150); plt.show()


In [ ]:
"""Train displacement network for one-step generation"""
print('Generating trajectory data...')
z0_train, traj_train, _ = generate_trajectories(velocity_model, n_samples=100000, n_steps=100, device=device)
z1_train = traj_train[-1]
displacement_target = (z1_train - z0_train).cpu(); z0_train_cpu = z0_train.cpu()
print(f'Generated {z0_train_cpu.size(0)} trajectory pairs')

class DisplacementMLP(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, input_dim))
    def forward(self, x):
        return self.net(x)

disp_model = DisplacementMLP(input_dim=2, hidden_dim=256).to(device)
optim_disp = torch.optim.Adam(disp_model.parameters(), lr=1e-3)
disp_dataset = torch.utils.data.TensorDataset(z0_train_cpu, displacement_target)
disp_loader = torch.utils.data.DataLoader(disp_dataset, batch_size=4096, shuffle=True)
total_epochs = 50; losses = AverageMeter('Loss', ':.6f')
for epoch in range(total_epochs):
    for z0_batch, target_batch in disp_loader:
        z0_batch = z0_batch.to(device); target_batch = target_batch.to(device)
        pred = disp_model(z0_batch)
        loss = torch.mean((pred - target_batch) ** 2)
        optim_disp.zero_grad(); loss.backward(); optim_disp.step()
        losses.update(loss.item())
    if (epoch + 1) % 10 == 0: print(f'Epoch {epoch+1}/{total_epochs}, Loss: {losses.avg:.6f}')
print('Finished Displacement Network Training')


In [ ]:
"""Compare 1-step Mean Flow vs 100-step ODE"""
fig, axes = plt.subplots(1, 3, figsize=(21, 7)); scatter_range = [-10, 10]; n_gen = 10000
gt = next(iter(torch.utils.data.DataLoader(EightGaussians_Dataset(total_len=n_gen), batch_size=n_gen)))
axes[0].scatter(gt[:, 0].numpy(), gt[:, 1].numpy(), s=1, alpha=0.5)
axes[0].set_xlim(scatter_range); axes[0].set_ylim(scatter_range); axes[0].set_title('Ground Truth')
z0 = torch.randn(n_gen, 2).to(device)
z1_ode, _ = euler_ode_solve(velocity_model, z0, n_steps=100)
s = z1_ode.detach().cpu().numpy()
axes[1].scatter(s[:, 0], s[:, 1], s=1, alpha=0.5)
axes[1].set_xlim(scatter_range); axes[1].set_ylim(scatter_range); axes[1].set_title('100-Step ODE')
with torch.no_grad(): z1_mf = z0 + disp_model(z0)
s = z1_mf.detach().cpu().numpy()
axes[2].scatter(s[:, 0], s[:, 1], s=1, alpha=0.5)
axes[2].set_xlim(scatter_range); axes[2].set_ylim(scatter_range); axes[2].set_title('1-Step Mean Flow')
plt.tight_layout(); plt.savefig('result/meanflow_one_step_comparison.png', dpi=150); plt.show()


### References

- Ho J, Jain A, Abbeel P. *Denoising diffusion probabilistic models*. NeurIPS, 2020.
- Lipman Y, Chen R T Q, Ben-Hamu H, et al. *Flow matching for generative modeling*. ICLR, 2023.
- Geng Z, Pokle A, Luo W, et al. *Mean Flows for one-step generative modeling*. arXiv preprint arXiv:2505.13447, 2025.
